In [ ]:
%pip install -U ddgs
from ddgs import DDGS

In [ ]:
%pip install easyocr
import easyocr
reader = easyocr.Reader(['en'])

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
DRIVE_PATH = '/content/gdrive/MyDrive/Scraped_Images'

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os
import io
import requests
import time

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

In [ ]:
df_1 = pd.read_parquet("/content/gdrive/My Drive/Colab Notebooks/test images/Food Ingredients Data/train-00000-of-00003.parquet")
df_2 = pd.read_parquet("/content/gdrive/My Drive/Colab Notebooks/test images/Food Ingredients Data/train-00001-of-00003.parquet")
df_3 = pd.read_parquet("/content/gdrive/My Drive/Colab Notebooks/test images/Food Ingredients Data/train-00002-of-00003.parquet")
df = pd.concat([df_1, df_2, df_3], axis=0)
df.head()

In [ ]:
queries = (df['ingredient'] + ' ' + df['subcategory'] + ' ' + df['category']).tolist()
queries = list(set(queries))
len(queries)

In [ ]:
def has_text(image_bytes):
  results = reader.readtext(io.BytesIO(image_bytes))
  if len(results) > 0:
    return True
  return False

In [ ]:
def scrape_images(query):
  count = 0
  dir_path = os.path.join(DRIVE_PATH, query)
  os.makedirs(dir_path, exist_ok=True)
  while count < 100:
    results = DDGS().images(query=query, max_results=100, region='us-en')
    for each in results:
      if count >= 100:
        break
      try:
        data = requests.get(each['image']).content
        if has_text(data):
          continue
        path = os.path.join(dir_path, f"{query}_{count}.jpg")
        with open(path, 'wb') as f:
          f.write(data)
        count += 1
      except Exception as e:
        continue
  with open('progress.txt', 'a') as f:
    f.write(f'{query} {count}\n')

In [ ]:
progress_file = 'progress.txt'
if os.path.exists(progress_file):
  with open(progress_file, 'r') as f:
    lines = f.read()
else:
  lines = ''
done = lines.split('\n')
for item in queries:
  if f'{item} 100' in done:
    continue
  scrape_images(item)
  time.sleep(180)